<a href="https://colab.research.google.com/github/munawirmt/Rainfall-weather-prediction-model/blob/main/rainfall_and_weather_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAIN FALL AND WEATHER PREDICTION MODEL

IMPORT LIBRERIES

In [1]:
import os
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor
from google.colab import drive

DATA LOADING & EXAMINING

In [ ]:
##load data
drive.mount("/content/drive")

def load_dataset(file_path):
    """Loads dataset efficiently from Excel file and standardizes column names."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found at path: {file_path}")

    excel_file = pd.ExcelFile(file_path)
    sheet_name = excel_file.sheet_names[0]

    df = pd.read_excel(excel_file, sheet_name=sheet_name)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    print(f"Loaded sheet '{sheet_name}' with shape: {df.shape}")
    return df
##exicution
DRIVE_FILE_PATH = "/content/drive/MyDrive/RAIN FALL AND WEATHER PREDICTION MODEL/india_weather_rainfall_data.xlsx"

df = load_dataset(DRIVE_FILE_PATH)


Mounted at /content/drive


***USING APACHE PARQUET FOR BIGDATA ANALYSIS***

In [ ]:
##Notebook conversion to parquet
excel_file = pd.ExcelFile(
    "/content/drive/MyDrive/RAIN FALL AND WEATHER PREDICTION MODEL/india_weather_rainfall_data.xlsx"
)
df = pd.read_excel(excel_file, sheet_name=excel_file.sheet_names[0])
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)
# Export to Apache Parquet
df.to_parquet(
    "/content/drive/MyDrive/RAIN FALL AND WEATHER PREDICTION MODEL/weather_predict_data.parquet",
    index=False,
    compression="snappy",
)
print("Dataset successfully converted to Parquet format.")


Dataset successfully converted to Parquet format.


In [ ]:
##main modelling notebook
from google.colab import drive
drive.mount("/content/drive")
# Professional, high-speed data loading pipeline
DATA_PATH = "/content/drive/MyDrive/RAIN FALL AND WEATHER PREDICTION MODEL/weather_predict_data.parquet"

def load_production_data(file_path):
    """Loads a production-scale Parquet dataset into memory instantly."""
    df = pd.read_parquet(file_path)
    print(f"Production dataset loaded with shape: {df.shape}")
    return df
df = load_production_data(DATA_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Production dataset loaded with shape: (970339, 15)


In [ ]:
df

,DATE_OF_RECORD,MONTH,SEASON,STATION_NAME,STATE,DISTRICT,AVG_TEMP,MIN_TEMP,MAX_TEMP,WIND_SPEED,AIR_PRESSURE,ELEVATION,LATITUDE,LONGITUDE,RAINFALL
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.0500,74.4000,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.0500,74.4000,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.0500,74.4000,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.0500,74.4000,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.0500,74.4000,25.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
970334,2025-02-06,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,27.3,22.0,34.0,7.6,1011.4,5,17.7167,83.2333,0.0
970335,2025-02-07,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.7,23.0,30.0,7.7,1013.1,5,17.7167,83.2333,0.3
970336,2025-02-08,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.7,1013.0,5,17.7167,83.2333,0.0
970337,2025-02-09,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.5,1012.8,5,17.7167,83.2333,0.0


In [ ]:
df.keys()

Index(['DATE_OF_RECORD', 'MONTH', 'SEASON', 'STATION_NAME', 'STATE',
       'DISTRICT', 'AVG_TEMP', 'MIN_TEMP', 'MAX_TEMP', 'WIND_SPEED',
       'AIR_PRESSURE', 'ELEVATION', 'LATITUDE', 'LONGITUDE', 'RAINFALL'],
      dtype='object')

In [ ]:
df.head()

,DATE_OF_RECORD,MONTH,SEASON,STATION_NAME,STATE,DISTRICT,AVG_TEMP,MIN_TEMP,MAX_TEMP,WIND_SPEED,AIR_PRESSURE,ELEVATION,LATITUDE,LONGITUDE,RAINFALL
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5


In [ ]:
df.tail()

,DATE_OF_RECORD,MONTH,SEASON,STATION_NAME,STATE,DISTRICT,AVG_TEMP,MIN_TEMP,MAX_TEMP,WIND_SPEED,AIR_PRESSURE,ELEVATION,LATITUDE,LONGITUDE,RAINFALL
970334,2025-02-06,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,27.3,22.0,34.0,7.6,1011.4,5,17.7167,83.2333,0.0
970335,2025-02-07,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.7,23.0,30.0,7.7,1013.1,5,17.7167,83.2333,0.3
970336,2025-02-08,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.7,1013.0,5,17.7167,83.2333,0.0
970337,2025-02-09,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,26.5,23.0,30.0,8.5,1012.8,5,17.7167,83.2333,0.0
970338,2025-02-10,February,Winter,Vizagapatam / Gajuwaka,AP,Visakhapatnam,25.5,21.8,31.0,8.2,1013.9,5,17.7167,83.2333,0.0


FEATURE ENGINEERING

In [ ]:
def engineer_rainfall_features(df):
    """Engineers seasonal aggregations and temporal features typical for Indian weather data."""
    df_feat = df.copy()
    # Identify Monthly Columns if present (JAN to DEC)
    months = [
        "JAN",
        "FEB",
        "MAR",
        "APR",
        "MAY",
        "JUN",
        "JUL",
        "AUG",
        "SEP",
        "OCT",
        "NOV",
        "DEC",
    ]
    present_months = [m for m in months if m in df_feat.columns]

    if len(present_months) == 12:
        # Define IMD Meteorological Seasons in India
        df_feat["WINTER_RAINFALL"] = df_feat[["JAN", "FEB"]].sum(axis=1)
        df_feat["PRE_MONSOON_RAINFALL"] = df_feat[
            ["MAR", "APR", "MAY"]
        ].sum(axis=1)
        df_feat["SW_MONSOON_RAINFALL"] = df_feat[
            ["JUN", "JUL", "AUG", "SEP"]
        ].sum(axis=1)
        df_feat["NE_MONSOON_RAINFALL"] = df_feat[
            ["OCT", "NOV", "DEC"]
        ].sum(axis=1)

        # Seasonal ratios and statistical properties
        df_feat["MONTHLY_STD"] = df_feat[present_months].std(axis=1)
        df_feat["MONTHLY_MAX"] = df_feat[present_months].max(axis=1)
        df_feat["MONTHLY_MIN"] = df_feat[present_months].min(axis=1)

        if "ANNUAL" not in df_feat.columns:
            df_feat["ANNUAL"] = df_feat[present_months].sum(axis=1)

    # Parse date components if a Date/Timestamp column exists
    date_cols = [c for c in df_feat.columns if "DATE" in c or "TIME" in c]
    if date_cols:
        date_col = date_cols[0]
        df_feat[date_col] = pd.to_datetime(df_feat[date_col], errors="coerce")
        df_feat["YEAR"] = df_feat[date_col].dt.year
        df_feat["MONTH"] = df_feat[date_col].dt.month
        df_feat["DAY_OF_YEAR"] = df_feat[date_col].dt.dayofyear
        df_feat.drop(columns=[date_col], inplace=True)

    return df_feat

DATA PREPROCESSING

In [ ]:
def build_preprocessor(X):
    """Builds scikit-learn preprocessing pipeline for numeric and categorical features."""
    numeric_features = X.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()
    categorical_features = X.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    return preprocessor

MODEL EVALUTION

In [ ]:
def evaluate_models(X_train, X_test, y_train, y_test, preprocessor):
    """Trains and benchmarks multiple regression models."""
    models = {
        "Random Forest": RandomForestRegressor(
            n_estimators=100, random_state=42
        ),
        "XGBoost": XGBRegressor(
            n_estimators=100, learning_rate=0.05, random_state=42
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }
    results = {}
    trained_pipelines = {}
    for name, model in models.items():
        pipeline = Pipeline(
            steps=[("preprocessor", preprocessor), ("regressor", model)]
        )

        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)

        results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
        trained_pipelines[name] = pipeline

        print(
            f"[{name}] -> R²: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}"
        )
    return results, trained_pipelines

HYPERPARAMETER TUNING FOR BEST MODEL

In [ ]:
def tune_best_model(X_train, y_train, preprocessor):
    """Performs RandomizedSearchCV tuning on XGBoost model."""
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", XGBRegressor(random_state=42)),
        ]
    )
    param_distributions = {
        "regressor__n_estimators": [100, 200, 300],
        "regressor__max_depth": [3, 5, 7, 10],
        "regressor__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "regressor__subsample": [0.7, 0.8, 1.0],
        "regressor__colsample_bytree": [0.7, 0.8, 1.0],
    }
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions,
        n_iter=15,
        cv=5,
        scoring="r2",
        random_state=42,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)

    print("\nBest Parameters Found:", search.best_params_)
    return search.best_estimator_

VISUALIZATION

In [ ]:
def plot_model_performance(y_test, predictions, model_name="Best Model"):
    """Plots actual vs predicted targets and residuals."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Actual vs Predicted Plot
    sns.scatterplot(
        x=y_test, y=predictions, ax=axes[0], color="#2b5c8f", alpha=0.6
    )
    axes[0].plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        "r--",
        lw=2,
    )
    axes[0].set_title(f"{model_name}: Actual vs Predicted")
    axes[0].set_xlabel("Actual Rainfall")
    axes[0].set_ylabel("Predicted Rainfall")

    # Residual Distribution
    residuals = y_test - predictions
    sns.histplot(residuals, kde=True, ax=axes[1], color="#e74c3c")
    axes[1].set_title("Residual Error Distribution")
    axes[1].set_xlabel("Residual (Actual - Predicted)")

    plt.tight_layout()
    plt.savefig("model_performance.png", dpi=300)
    plt.show()

MAIN EXECUTION PIPELINE

In [ ]:
def main():
    # FIXED: Replaced relative path string with the absolute Google Drive path variable
    file_path = DRIVE_FILE_PATH

    # Step 1: Load and Clean
    df_raw = load_dataset(file_path)

    # Step 2: Feature Engineering
    df = engineer_rainfall_features(df_raw)

    # Step 3: Determine Target & Features
    target_col = None
    possible_targets = [
        "ANNUAL",
        "RAINFALL",
        "ANNUAL_RAINFALL",
        "SUBDIVISION_RAINFALL",
    ]
    for target in possible_targets:
        if target in df.columns:
            target_col = target
            break

    if target_col is None:
        # Fallback to last numerical column as target
        num_cols = df.select_dtypes(include=[np.number]).columns
        target_col = num_cols[-1]

    print(f"Target column selected: '{target_col}'")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Handle target NaNs if any
    valid_idx = y.notna()
    X = X[valid_idx]
    y = y[valid_idx]

    # Step 4: Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Step 5: Build Preprocessor
    preprocessor = build_preprocessor(X)

    # Step 6: Benchmark Base Models
    print("\n--- Benchmarking Base Models ---")
    results, pipelines = evaluate_models(
        X_train, X_test, y_train, y_test, preprocessor
    )

    # Step 7: Hyperparameter Optimization on Top Architecture
    print("\n--- Tuning Hyperparameters for Best Model (XGBoost) ---")
    tuned_model = tune_best_model(X_train, y_train, preprocessor)

    # Step 8: Final Evaluation & Plotting
    final_preds = tuned_model.predict(X_test)
    final_r2 = r2_score(y_test, final_preds)
    final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))

    print(
        f"\nFinal Tuned Model Performance -> R²: {final_r2:.4f} | RMSE: {final_rmse:.4f}"
    )

    plot_model_performance(y_test, final_preds, model_name="Tuned XGBoost")

    # Step 9: Save Pipeline
    # OPTIMIZATION: Save the model directly into Google Drive so it isn't lost when Colab resets
    output_model_path = os.path.join(
        os.path.dirname(DRIVE_FILE_PATH), "best_rainfall_model.pkl"
    )
    joblib.dump(tuned_model, output_model_path)
    print(f"\nModel saved successfully at '{output_model_path}'.")


if __name__ == "__main__":
    main()


Loaded sheet 'Sheet1' with shape: (970339, 15)
Target column selected: 'RAINFALL'

--- Benchmarking Base Models ---
